# TTC Subway Delays by Category — Monthly View
Visualises pre-aggregated data from `data/prep/ttc-subway-delay-data/monthly_by_category.csv`.  
Covers January 2014 – present. Note: 2018–2021 files in the source data contain January only.

In [1]:
from pathlib import Path

import polars as pl
import altair as alt

In [2]:
PREP = Path("../data/prep/ttc-subway-delay-data/monthly_by_category.csv")

df = pl.read_csv(PREP).with_columns(
    pl.col("year_month").cast(pl.Utf8).str.to_date("%Y%m").alias("date"),
)

print(f"Rows: {len(df)}  |  Months: {df['year_month'].n_unique()}  |  Categories: {df['category'].n_unique()}")
print(f"Range: {df['date'].min()} – {df['date'].max()}")

PALETTE = {
    "Mechanical/Infrastructure": "#4C72B0",
    "Mechanical/Vehicle":        "#DD8452",
    "Operational/Process":       "#55A868",
    "Patron":                    "#C44E52",
    "Security":                  "#E377C2",
    "Weather":                   "#8172B3",
    "Other":                     "#937860",
    # "Other/External":            "#DA8BC3",
    # "Other/Internal":            "#8C8C8C",
    # "Planned Work":              "#CCB974",
    # "Unplanned Work":            "#64B5CD",
}
domain = list(PALETTE.keys())
palette_range = list(PALETTE.values())
color_scale = alt.Scale(domain=domain, range=palette_range)

Rows: 630  |  Months: 94  |  Categories: 7
Range: 2014-01-01 – 2026-01-01


## 1. Delay Incidents over Time by Category

In [3]:
alt.Chart(df).mark_line().encode(
    x=alt.X("date:T", title="Month"),
    y=alt.Y("delays:Q", title="Delay Incidents"),
    color=alt.Color("category:N", title="Category", scale=color_scale),
    tooltip=[
        alt.Tooltip("date:T", title="Month", format="%b %Y"),
        "category",
        "delays",
    ],
).properties(
    title="Monthly Delay Incidents by Category",
    width=850, height=300,
)

alt.Chart(...)

## 2. Total Delay Minutes over Time by Category

In [4]:
alt.Chart(df).mark_line().encode(
    x=alt.X("date:T", title="Month"),
    y=alt.Y("total_delay:Q", title="Total Delay (minutes)"),
    color=alt.Color("category:N", title="Category", scale=color_scale),
    tooltip=[
        alt.Tooltip("date:T", title="Month", format="%b %Y"),
        "category",
        alt.Tooltip("total_delay:Q", title="Total delay (min)"),
    ],
).properties(
    title="Monthly Total Delay Minutes by Category",
    width=850, height=300,
)

alt.Chart(...)

## 3. Major Delays over Time by Category

In [5]:
alt.Chart(df).mark_line().encode(
    x=alt.X("date:T", title="Month"),
    y=alt.Y("major_delays:Q", title="Major Delays (≥20 min)"),
    color=alt.Color("category:N", title="Category", scale=color_scale),
    tooltip=[
        alt.Tooltip("date:T", title="Month", format="%b %Y"),
        "category",
        "major_delays",
    ],
).properties(
    title="Monthly Major Delays (≥20 min) by Category",
    width=850, height=300,
)

alt.Chart(...)

## 4. Stacked Area — Composition over Time

Shows how the share of each category shifts across the full history.

In [6]:
alt.Chart(df).mark_area().encode(
    x=alt.X("date:T", title="Month"),
    y=alt.Y("delays:Q", stack="normalize", title="Share of Incidents"),
    color=alt.Color("category:N", title="Category", scale=color_scale),
    tooltip=[
        alt.Tooltip("date:T", title="Month", format="%b %Y"),
        "category",
        "delays",
    ],
).properties(
    title="Category Share of Incidents over Time (normalised)",
    width=850, height=280,
)

alt.Chart(...)

## 5. Recent 24 Months — All Three Metrics

Zooms in on the most recent two years for a clearer view of current trends.

In [7]:
cutoff = df["date"].max().replace(year=df["date"].max().year - 2)
recent = df.filter(pl.col("date") >= cutoff)

base = alt.Chart(recent).encode(
    x=alt.X("date:T", title="Month"),
    color=alt.Color("category:N", title="Category", scale=color_scale),
)

r1 = base.mark_line(point=True).encode(
    y=alt.Y("delays:Q", title="Incidents"),
    tooltip=[alt.Tooltip("date:T", format="%b %Y"), "category", "delays"],
).properties(title="Delay Incidents", width=820, height=200)

r2 = base.mark_line(point=True).encode(
    y=alt.Y("total_delay:Q", title="Total delay (min)"),
    tooltip=[alt.Tooltip("date:T", format="%b %Y"), "category", alt.Tooltip("total_delay:Q", title="Total min")],
).properties(title="Total Delay Minutes", width=820, height=200)

r3 = base.mark_line(point=True).encode(
    y=alt.Y("major_delays:Q", title="Major delays"),
    tooltip=[alt.Tooltip("date:T", format="%b %Y"), "category", "major_delays"],
).properties(title="Major Delays (≥20 min)", width=820, height=200)

alt.vconcat(r1, r2, r3).properties(spacing=10)

alt.VConcatChart(...)

## 6. Heatmap — Category × Month (recent 24 months)

Highlights which categories spike in which months.

In [8]:
alt.Chart(recent).mark_rect().encode(
    x=alt.X("date:T", title="Month", timeUnit="yearmonth"),
    y=alt.Y("category:N", sort=domain, title=""),
    color=alt.Color("delays:Q", title="Incidents", scale=alt.Scale(scheme="blues")),
    tooltip=[
        alt.Tooltip("date:T", title="Month", format="%b %Y"),
        "category",
        "delays",
        "total_delay",
        "major_delays",
    ],
).properties(
    title="Delay Incidents Heatmap — Category × Month (recent 24 months)",
    width=820, height=260,
)

alt.Chart(...)